In [0]:
from pyspark.sql.functions import col, when, timestamp_diff
from datetime import date
from dateutil.relativedelta import relativedelta

In [0]:
# Define the start date of the target month (six months ago)
target_month_start = date.today().replace(day=1) - relativedelta(months=6)

# Define the start date of the following month (used as the upper filter bound)
next_month_start = target_month_start + relativedelta(months=1)

In [0]:
# Read the "yellow_trips_raw" table from the "nyctaxi.01_bronze" schema
# Then filter rows where "tpep_pickup_datetime" is >= six months ago start and < one month ago start (i.e., only the month that is six months before today)

df = spark.read.table("nyctaxi.01_bronze.yellow_trips_raw").filter(f"tpep_pickup_datetime >= '{target_month_start}' AND tpep_pickup_datetime < '{next_month_start}'")

In [0]:
# Selects required taxi trip columns while calculating trip duration in minutes
# Decodes numeric IDs (Vendor, Ratecode, Payment) into human-readable text descriptions
df = df.select(
    when(col("VendorID") == 1, "Creative Mobile Technologies, LLC")
    .when(col("VendorID") == 2, "Curb Mobility, LLC")
    .when(col("VendorID") == 6, "Myle Technologies Inc")
    .when(col("VendorID") == 7, "Helix")
    .otherwise("Unknown")
    .alias("vendor"),

    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    timestamp_diff('MINUTE', df.tpep_pickup_datetime, df.tpep_dropoff_datetime).alias("trip_duration"),
    "passenger_count",
    "trip_distance",

    when(col("RatecodeID") == 1, "Standard rate")
    .when(col("RatecodeID") == 2, "JFK")
    .when(col("RatecodeID") == 3, "Newark")
    .when(col("RatecodeID") == 4, "Nassau or Westchester")
    .when(col("RatecodeID") == 5, "Negotiated fare")
    .when(col("RatecodeID") == 6, "Group ride")
    .otherwise("Unknown")
    .alias("rate_type"),

    "store_and_fwd_flag",
    col("PULocationID").alias("pu_location_id"),
    col("DOLocationID").alias("do_location_id"),
    
    when(col("payment_type") == 1, "Flex Fare trip")
    .when(col("payment_type") == 2, "Credit card")
    .when(col("payment_type") == 3, "No charge")
    .when(col("payment_type") == 4, "Dispute")
    .when(col("payment_type") == 6, "Voided trip")
    .otherwise("Unknown")
    .alias("payment_type"),

    "fare_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    col("Airport_fee").alias("airport_fee"),
    "cbd_congestion_fee",
    "processed_timestamp"
)


In [0]:
# Write the fully cleansed and transformed taxi trip data into the Silver layer.
# Overwrites the target Delta table to refresh it with the finalized structure and columns.
df.write.mode("append").saveAsTable("nyctaxi.02_silver.yellow_trips_cleansed")